clone the code and have env setup.

In [1]:
import os

REPO_URL = 'https://github.com/Lynx-Zhang/DD2424-Project.git'
REPO_DIR = '/content/DD2424-Project'
# BRANCH = 'dev/AdamAndSentiment'
BRANCH = 'main'


from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}

print(f"\n当前分支: ", end='')
!git branch --show-current



!mkdir -p predictions
!mkdir -p /content/drive/MyDrive/gpt2_logs
!mkdir -p /content/drive/MyDrive/gpt2_checkpoints

print("\n GPU 信息:")

print("\n 环境准备完成！")


Mounted at /content/drive
Cloning into '/content/DD2424-Project'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 87 (delta 27), reused 35 (delta 22), pack-reused 30 (from 1)
Receiving objects: 100% (87/87), 31.92 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/DD2424-Project

当前分支: dev/AdamAndSentiment

🖥 GPU 信息:

✅ 环境准备完成！


For Sentiment Analysis Part we have

In [2]:
# 按照 cs224n_dfp 环境装依赖（pip 版本）
!pip install -q \
    tqdm==4.58.0 \
    requests==2.25.1 \
    importlib-metadata==3.7.0 \
    filelock==3.0.12 \
    tokenizers==0.20 \
    explainaboard_client==0.0.7 \
    einops==0.8.0 \
    transformers==4.46.3 \
    sacrebleu==2.5.1 \
    scikit-learn

# torch 不动（用 Colab 自带的 CUDA 版）
print(" 依赖安装完成")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [3]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-31cc4152-2809-fe73-faf6-55375998d419)


In [4]:
!pwd
! ls
%cd DD2424-Project
! ls


/content/DD2424-Project
classifier.py					 optimizer.py
config.py					 optimizer_test.npy
CS_224n__Default_Final_Project__Build_GPT_2.pdf  optimizer_test.py
data						 paraphrase_detection.py
datasets.py					 predictions
doc						 prepare_submit.py
env.yml						 README.md
evaluation.py					 sanity_check.py
LICENSE						 setup.sh
models						 sonnet_generation.py
modules						 utils.py
[Errno 2] No such file or directory: 'DD2424-Project'
/content/DD2424-Project
classifier.py					 optimizer.py
config.py					 optimizer_test.npy
CS_224n__Default_Final_Project__Build_GPT_2.pdf  optimizer_test.py
data						 paraphrase_detection.py
datasets.py					 predictions
doc						 prepare_submit.py
env.yml						 README.md
evaluation.py					 sanity_check.py
LICENSE						 setup.sh
models						 sonnet_generation.py
modules						 utils.py


In [5]:
# 跑 last-linear-layer 模式
!python classifier.py --use_gpu --fine-tune-mode last-linear-layer --epochs 10 --lr 1e-3 --batch_size 64
# 备份到 Drive
!cp -r predictions /content/drive/MyDrive/gpt2_checkpoints/last-linear-$(date +%H%M%S)



2026-05-02 06:15:21.045280: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
eval: 100%|██████████| 61/61 [00:13<00:00,  4.68it/s]
Training Sentiment Classifier on SST...
load 8544 data from data/ids-sst-train.csv
load 1101 data from data/ids-sst-dev.csv
save the model to sst-classifier.pt
Epoch 0: train loss :: 1.656, train acc :: 0.416, dev acc :: 0.406
save the model to sst-classifier.pt
Epoch 1: train loss :: 1.412, train acc :: 0.433, dev acc :: 0.418
Epoch 2: train loss :: 1.365, train acc :: 0.377, dev acc :: 0.368
save the model to sst-classifier.pt
Epoch 3: train loss :: 1.329, train acc :: 0.463, dev acc :: 0.445
Epoch 4: train loss :: 1.309, train acc :: 0.448, dev acc :: 0.426
Epoch 5: train loss :: 1.297, train acc :: 0.471, dev acc :

In [8]:
# 跑 full-model 模式
!python classifier.py --use_gpu --fine-tune-mode full-model --epochs 10 --lr 1e-5 --batch_size 16

# 备份到 Drive
!cp -r predictions /content/drive/MyDrive/gpt2_checkpoints/full-model-$(date +%H%M%S)

2026-05-02 06:52:41.397542: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Training Sentiment Classifier on SST...
load 8544 data from data/ids-sst-train.csv
load 1101 data from data/ids-sst-dev.csv
train-0: 100% 534/534 [01:38<00:00,  5.42it/s]
eval: 100% 534/534 [00:23<00:00, 23.09it/s]
eval: 100% 69/69 [00:03<00:00, 22.98it/s]
save the model to sst-classifier.pt
Epoch 0: train loss :: 1.492, train acc :: 0.496, dev acc :: 0.452
train-1: 100% 534/534 [01:38<00:00,  5.40it/s]
eval: 100% 534/534 [00:23<00:00, 22.87it/s]
eval: 100% 69/69 [00:02<00:00, 23.81it/s]
save the model to sst-classifier.pt
Epoch 1: train loss :: 1.182, train acc :: 0.566, dev acc :: 0.513
train-2: 100% 534/534 [01:38<00:00,  5.43it/s]
eval: 100% 534/534 [00:23<00:00, 23.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')